# Monitor Azure AI Foundry — Log Analytics Queries

Query a Log Analytics workspace to retrieve **AzureMetrics** data for Azure AI Foundry (Cognitive Services) resources.

Requirements:
- `azure-identity` and `azure-monitor-query` Python packages
- A Log Analytics workspace with diagnostic settings streaming **AllMetrics** from your Foundry instances
- Azure CLI login (`az login`) or another credential available to `DefaultAzureCredential`

Reference: [Monitor Azure AI Foundry models](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/how-to/monitor-models?view=foundry-classic)

In [ ]:
%pip install azure-identity azure-monitor-query pandas python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# id of the log analytics workspace where your foundry instance is sending logs to
# Find it in the Azure Portal → Log Analytics workspace → Properties → Workspace ID
WORKSPACE_ID = os.environ["WORKSPACE_ID"]
# location of your Cognitive Services account, e.g. "westus2", "westeurope", "swedencentral"
LOCATION = os.environ["LOCATION"]
# ── Set the full resource ID of your foundry instance ─────
# Format: /subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{name}
RESOURCE_ID = os.environ["RESOURCE_ID"]
# Azure Resource Manager API version
API_VERSION = os.environ["API_VERSION"]

In [ ]:
from datetime import timedelta

from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
import pandas as pd

credential = DefaultAzureCredential()
client = LogsQueryClient(credential)

# ── Set your Log Analytics workspace ID here ────────────────────────

query = """
AzureMetrics
| union AzureDiagnostics
| take 2
"""

response = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=1))

if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df = pd.DataFrame(data=table.rows, columns=table.columns)
    print("Query succeeded, here are the results:")
    display(df)
else:
    print("Query failed:", response.partial_error)

## Query Azure Monitor Metrics directly

Use `MetricsQueryClient` to query metrics straight from a Cognitive Services resource — no Log Analytics workspace required.

This queries the Azure Monitor Metrics API directly, which gives you near-real-time data with up to 1-minute granularity.

Reference: [azure-monitor-query MetricsQueryClient](https://learn.microsoft.com/en-us/python/api/azure-monitor-query/azure.monitor.query.metricsqueryclient)

In [ ]:
%pip install azure-monitor-querymetrics --quiet

In [ ]:
from azure.monitor.querymetrics import MetricsClient, MetricAggregationType

# The MetricsClient needs the regional Azure Monitor endpoint for your resource's region.
# Format: https://<region>.metrics.monitor.azure.com
METRICS_ENDPOINT = f"https://{LOCATION}.metrics.monitor.azure.com"

metrics_client = MetricsClient(endpoint=METRICS_ENDPOINT, credential=credential)


### Total tokens (direct metrics, last 24h)

In [ ]:
results = metrics_client.query_resources(
    resource_ids=[RESOURCE_ID],
    metric_namespace="Microsoft.CognitiveServices/accounts",
    metric_names=["TotalTokens", "ProcessedPromptTokens", "GeneratedTokens"],
    timespan=timedelta(hours=1),
    granularity=timedelta(hours=1),
    aggregations=[MetricAggregationType.TOTAL],
    filter="ModelDeploymentName eq '*'",  # Split by model deployment (like "Apply splitting" in the Portal)
)

rows = []
for result in results:
    for metric in result.metrics:
        for ts_element in metric.timeseries:
            deployment = ts_element.metadata_values.get("modeldeploymentname", "")
            for metric_value in ts_element.data:
                if metric_value.total is not None:
                    rows.append({
                        "Timestamp": metric_value.timestamp,
                        "Metric": metric.name,
                        "ModelDeploymentName": deployment,
                        "Total": metric_value.total,
                    })

df_tokens = pd.DataFrame(rows)

print("Metrics query succeeded, here are the results:")
print("Metrics for resource:", RESOURCE_ID)
display(df_tokens)

### Same data from AzureMetrics in Log Analytics

Check whether the same per-deployment token breakdown is available via the `AzureMetrics` table in the Log Analytics workspace.

In [ ]:
# Compare: AzureMetrics in Log Analytics vs direct Metrics API
# First, see which token-related metrics are available
query = """
AzureMetrics
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where MetricName has "Token" or MetricName in ("ProcessedPromptTokens", "GeneratedTokens")
| summarize
    Total = sum(Total),
    Rows = count(),
    MinTime = min(TimeGenerated),
    MaxTime = max(TimeGenerated)
    by Resource, MetricName
| order by Resource asc, MetricName asc
"""

response = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=7))

if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df_la_tokens = pd.DataFrame(data=table.rows, columns=table.columns)
    print(f"Token metrics in AzureMetrics (Log Analytics) — {len(df_la_tokens)} rows:")
    display(df_la_tokens)
    
    if len(df_la_tokens) == 0:
        print("\n⚠ No token metrics found in AzureMetrics.")
        print("This means the diagnostic settings are NOT streaming token metrics to Log Analytics.")
        print("Only the direct Azure Monitor Metrics API has this data.")
else:
    print("Query failed:", response.partial_error)

### AzureDiagnostics — per-request detail

`AzureDiagnostics` contains one row per API call, with deployment name, model name, and request/response sizes.

In [ ]:
# AzureDiagnostics — per-request fields for Cognitive Services
query = """
AzureDiagnostics
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where TimeGenerated > ago(1d)
| extend props = parse_json(properties_s)
| extend
    deployment = tostring(props.modelDeploymentName),
    model = tostring(props.modelName),
    operation = OperationName,
    reqBytes = tolong(props.requestLength),
    respBytes = tolong(props.responseLength),
    statusCode = tostring(props.statusCode),
    apiName = tostring(props.apiName),
    region = tostring(props.modelRegion)
| project TimeGenerated, Resource, operation, deployment, model, region, statusCode, reqBytes, respBytes, apiName, properties_s
| take 10
"""

response = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(days=1))

if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df_diag = pd.DataFrame(data=table.rows, columns=table.columns)
    print(f"AzureDiagnostics — {len(df_diag)} sample rows")
    display(df_diag.drop(columns=["properties_s"]))
    
    if len(df_diag) > 0:
        print("\n── Raw properties_s for first row ──")
        import json
        print(json.dumps(json.loads(df_diag.iloc[0]["properties_s"]), indent=2))
    else:
        print("\n⚠ No AzureDiagnostics rows found. RequestResponse logs may not be enabled in diagnostic settings.")
else:
    print("Query failed:", response.partial_error)

### Quota & deployment allocation

Query the subscription-level quota limits per model/region, and the TPM allocation per deployment across all Cognitive Services accounts.

In [ ]:
import requests

SUBSCRIPTION_ID = RESOURCE_ID.split("/")[2]

token = credential.get_token("https://management.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}

# ── 1. Subscription-level quota per model in this region ────────────
usages_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/providers/Microsoft.CognitiveServices/locations/{LOCATION}"
    f"/usages?api-version={API_VERSION}"
)
usages_resp = requests.get(usages_url, headers=headers)
usages_resp.raise_for_status()

usages = usages_resp.json().get("value", [])
quota_rows = []
for u in usages:
    if u.get("currentValue", 0) > 0 or "OpenAI" in u.get("name", {}).get("value", ""):
        quota_rows.append({
            "Model": u["name"]["value"],
            "Allocated (current)": u["currentValue"],
            "Limit": u["limit"],
            "Unit": u.get("unit", ""),
        })

df_quota = pd.DataFrame(quota_rows)
print(f"Subscription quota in {LOCATION} — {len(df_quota)} model SKUs with allocation:")
display(df_quota)

# ── 2. Per-deployment allocation across all accounts ────────────────
# List all Cognitive Services accounts in the subscription
accounts_url = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}"
    f"/providers/Microsoft.CognitiveServices/accounts"
    f"?api-version={API_VERSION}"
)
accounts_resp = requests.get(accounts_url, headers=headers)
accounts_resp.raise_for_status()

deploy_rows = []
for acct in accounts_resp.json().get("value", []):
    acct_id = acct["id"]
    acct_name = acct["name"]
    acct_location = acct["location"]
    # List deployments for this account
    deploys_url = (
        f"https://management.azure.com{acct_id}"
        f"/deployments?api-version={API_VERSION}"
    )
    deploys_resp = requests.get(deploys_url, headers=headers)
    if deploys_resp.status_code != 200:
        continue
    for d in deploys_resp.json().get("value", []):
        props = d.get("properties", {})
        model = props.get("model", {})
        sku = d.get("sku", {})
        rate_limits = props.get("rateLimits", [])
        tpm = next((r["count"] for r in rate_limits if r.get("key") == "token"), sku.get("capacity"))
        rpm = next((r["count"] for r in rate_limits if r.get("key") == "request"), None)
        deploy_rows.append({
            "Account": acct_name,
            "Location": acct_location,
            "Deployment": d["name"],
            "Model": model.get("name", ""),
            "Version": model.get("version", ""),
            "SKU": sku.get("name", ""),
            "Capacity (TPM K)": sku.get("capacity", ""),
            "TPM (rate limit)": tpm,
            "RPM (rate limit)": rpm,
        })

df_deployments = pd.DataFrame(deploy_rows)
print(f"\nDeployments across all accounts — {len(df_deployments)} total:")
display(df_deployments)